# AdaBoost-FKD example

In this notebook, we present how to execute the AdaBoost-FKD algoritm, as well as the local version without the federated process, for comparison purposes.
It is assumed that with this example, the rest of experiments could be also addressed.

In [1]:
# Import data from flextrees
# TO-DO: Habria que poner algun ejemplo que no use dataset de flex?
from flextrees.datasets.tabular_datasets import adult
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

# Import AdaBoostFKD, and also the Federated Random Forest, for comparison
from models.AdaBoostFKD import AdaBoostFKD
from models.FRF import FRF_eval

from sklearn.datasets import load_breast_cancer

In [2]:
# Fix the random numbers seed
seed = 0

Load the dataset.

Note that the whole dataset (both train and test) is loaded and joined together. Later, the whole data is distributed among the clients to simulate the federated scenario.

In [3]:
train_data, test_data = adult(ret_feature_names=False, categorical=False)
X_data,y_data = train_data.to_numpy()
X_test,y_test = test_data.to_numpy()
X_data = np.concatenate((X_data,X_test))
y_data = np.concatenate((y_data,y_test))

#Alternatively, you can directly use any numpy dataset of your own:

#bcancer = load_breast_cancer()
#X_data = bcancer.data
#y_data = bcancer.target

In [4]:
# Separate public data (a small portion, for example, 5%)
# It is considered to be unlabeled, so we do not store the targets of the public data
data, public_data, targets, _ = train_test_split(X_data, y_data, test_size=0.05, random_state=seed)

# Get global test set (i.e., 10%), and the rest of training data that will be later distributed among clients
X_train, X_test, y_train, y_test = train_test_split(data,targets,test_size=0.1,random_state=seed)

Run AdaBoost-FKD

In [5]:
# When creating the model the data partition according to the chosen data distribution is made. 
fl_model = AdaBoostFKD(X_train, y_train, public_data, 
                         n_clients=10, T=10,
                         data_distribution='niid_quantity_skew', distribution_param=0.5,
                         public_data_prediction='weighted_majority_voting', 
                         server_alpha_weight_adj='common_weighted',
                         prediction_weights='only_server', 
                         random_state=seed,
                         server_classifier=DecisionTreeClassifier, server_classifier_params={'random_state':seed, 'max_depth':None, 'max_leaf_nodes':None},
                         clients_classifier=DecisionTreeClassifier, clients_classifier_params={'random_state':seed, 'max_depth':None, 'max_leaf_nodes':None})

# Store data distrib for subsequent models
train_dict = fl_model.train_clients_data.copy()
test_dict = fl_model.test_clients_data.copy()

# Takes the columns of weights of AdaBoost out
for key,(train,labeltr) in train_dict.items():
    train_dict[key] = (train[:,:-1],labeltr)
    test,labelte = test_dict[key]
    test_dict[key] = (test[:,:-1],labelte)

In [6]:
#Train the model
fl_model.fitmodel()

In [7]:
# Evaluate how the federated model works in local tests and global test (in average among clients) 
fl_acc_global, fl_f1_global, fl_acc_local, fl_f1_local = fl_model.overall_score(X_test, y_test)

print(f'AdaBoost-FKD accuracy (global test): {fl_acc_global:.4f}')
print(f'AdaBoost-FKD f1-score (global test): {fl_f1_global:.4f}')
print(f'AdaBoost-FKD accuracy (local test): {fl_acc_local:.4f}')
print(f'AdaBoost-FKD f1-score (local test): {fl_f1_local:.4f}')

AdaBoost-FKD accuracy (global test): 0.8332
AdaBoost-FKD f1-score (global test): 0.8398
AdaBoost-FKD accuracy (local test): 0.8378
AdaBoost-FKD f1-score (local test): 0.8520


Run local AdaBoost models at each client

In [8]:
# If we want to compare it to the local models (without federated process), we first need to train local models with same data
fl_model.fit_local_clients_models()

In [9]:
# Then we can get a Dataframe with all the results for each client
localAB_acc_scores = fl_model.overall_acc_score(X_test, y_test)
localAB_acc_scores

,data_distrib,FL_acc_own_data,FL_acc_global_data,local_acc_own_data,local_acc_global_data,local_difference,global_difference
0,1748.0,0.844394,0.833226,0.803204,0.801228,0.041190,0.031997
1,9016.0,0.850111,0.833226,0.815965,0.820621,0.034146,0.012605
2,208.0,0.792453,0.833226,0.773585,0.798319,0.018868,0.034906
3,673.0,0.863905,0.833226,0.834320,0.790239,0.029586,0.042986
4,1835.0,0.843137,0.833226,0.812636,0.796057,0.030501,0.037169
5,3911.0,0.845603,0.833226,0.808793,0.795410,0.036810,0.037815
6,19.0,0.800000,0.833226,0.600000,0.658694,0.200000,0.174531
7,3036.0,0.856390,0.833226,0.816864,0.823206,0.039526,0.010019
8,827.0,0.859903,0.833226,0.830918,0.790562,0.028986,0.042663
9,989.0,0.822581,0.833226,0.774194,0.788946,0.048387,0.044279


In [10]:
localAB_f1_scores = fl_model.overall_F1_score(X_test,y_test)
localAB_f1_scores

,data_distrib,FL_wf1_own_data,FL_wf1_global_data,local_wf1_own_data,local_wf1_global_data,local_difference_w,global_difference_w
0,1748.0,0.847446,0.8398,0.801159,0.801551,0.046286,0.038249
1,9016.0,0.855814,0.8398,0.813808,0.820253,0.042006,0.019547
2,208.0,0.805260,0.8398,0.758034,0.793568,0.047225,0.046232
3,673.0,0.862036,0.8398,0.834320,0.789809,0.027717,0.049991
4,1835.0,0.850280,0.8398,0.814743,0.799116,0.035537,0.040684
5,3911.0,0.853406,0.8398,0.809501,0.793564,0.043905,0.046236
6,19.0,0.888889,0.8398,0.566667,0.653062,0.322222,0.186738
7,3036.0,0.860892,0.8398,0.814893,0.824424,0.045999,0.015376
8,827.0,0.867904,0.8398,0.833207,0.791547,0.034697,0.048252
9,989.0,0.828191,0.8398,0.766149,0.785099,0.062042,0.054701


In [11]:
# Evaluate how the local models works in local tests and global test (in average among them) 
localAB_acc_global = localAB_acc_scores['local_acc_global_data'].mean()
localAB_f1_global = localAB_f1_scores['local_wf1_global_data'].mean()
localAB_acc_local = localAB_acc_scores['local_acc_own_data'].mean()
localAB_f1_local = localAB_f1_scores['local_wf1_own_data'].mean()

print(f'AdaBoost-FKD accuracy (global test): {localAB_acc_global:.4f}')
print(f'AdaBoost-FKD f1-score (global test): {localAB_f1_global:.4f}')
print(f'AdaBoost-FKD accuracy (local test): {localAB_acc_local:.4f}')
print(f'AdaBoost-FKD f1-score (local test): {localAB_f1_local:.4f}')

AdaBoost-FKD accuracy (global test): 0.7863
AdaBoost-FKD f1-score (global test): 0.7852
AdaBoost-FKD accuracy (local test): 0.7870
AdaBoost-FKD f1-score (local test): 0.7812


Compare to state-of-the-art methods

In [12]:
# Now we can use it to compare it to other state of the art algorithms such as FRF with 100 estimators
FRF_acc_global, FRF_f1_global, FRF_acc_local, FRF_f1_local = FRF_eval(train_dict, test_dict, X_test, y_test, hyperparameters='theirs')

In [13]:
print(f'FRF model accuracy (global test): {FRF_acc_global:.4f}')
print(f'FRF model f1-score (global test): {FRF_f1_global:.4f}')
print(f'FRF model accuracy (local test): {FRF_acc_local:.4f}')
print(f'FRF model f1-score (local test): {FRF_f1_local:.4f}')

FRF model accuracy (global test): 0.8394
FRF model f1-score (global test): 0.8190
FRF model accuracy (local test): 0.8395
FRF model f1-score (local test): 0.8151
